# Class 18 — First experiments with scikit-learn: Titanic

This notebook introduces the common scikit-learn interface **before** using Weights & Biases (W&B). We start from the Titanic dataset prepared in Classes 16/15 to answer this question: how can we train, transform data, predict, and compare candidates reproducibly?

By the end, you should be able to:

- distinguish estimators, transformers, and predictors;
- use `fit`, `transform`, `predict`, and `predict_proba`;
- build a `ColumnTransformer` for numeric and categorical data;
- wrap preprocessing and a model in a `Pipeline`;
- compare linear, neighbor-based, and tree-based model families without leaking test information.

> **Scope:** these are small, local experiments. The next activity can log parameters, metrics, and artifacts in W&B.

## 1. Setup and data

`Class15_Titanic_Imputed.csv` contains 891 passengers and was already curated in the previous classes. We therefore do not repeat the imputation decisions here; instead, we learn how to encode categories and scale numbers within the modeling workflow. The target is `survived` (1 = survived).

In [1]:
from pathlib import Path
import pickle

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# A fixed seed makes data splitting and stochastic models reproducible.
RANDOM_STATE = 42
DATA_PATH = Path("Class15_Titanic_Imputed.csv")

df = pd.read_csv(DATA_PATH)
print(f"Rows and columns: {df.shape}")
df.head()

Rows and columns: (891, 8)


,sex,pclass,embarked,age,fare,sibsp,parch,survived
0,male,3,S,22.0,7.2500,1.0,0.0,0
1,female,1,C,38.0,71.2833,1.0,0.0,1
2,female,3,S,26.0,7.9250,0.0,0.0,1
3,female,1,S,35.0,53.1000,1.0,0.0,1
4,male,3,S,35.0,8.0500,0.0,0.0,0


In [2]:
# `target` is excluded from X because it is the value the classifier must learn to predict.
target = "survived"
X = df.drop(columns=target)
y = df[target]

print("Feature data types:")
display(X.dtypes)
print("\nTarget distribution:")
display(y.value_counts(normalize=True))

Feature data types:


sex             str
pclass        int64
embarked        str
age         float64
fare        float64
sibsp       float64
parch       float64
dtype: object


Target distribution:


survived
0    0.616162
1    0.383838
Name: proportion, dtype: float64

## 2. One interface, different roles

scikit-learn uses conventions shared by many classes:

| Role | Main methods | Example |
| --- | --- | --- |
| **Estimator** | `fit(X, y)` | learns parameters from data |
| **Transformer** | `fit(X)` and `transform(X)` | `StandardScaler`, `OneHotEncoder` |
| **Predictor** | `fit(X, y)` and `predict(X)` | `LogisticRegression`, `RandomForestClassifier` |
| **Meta-estimator** | same interface, wrapping other objects | `Pipeline`, `GridSearchCV` |

The most important rule is: **fit only on training data; apply the exact same fitted object with `transform`/`predict` to new data.**

### 2.1 Train/test split

The test set stays untouched while we explore candidates. `stratify=y` approximately preserves the proportion of survivors in both subsets. `test_size=0.25` reserves 25% of the observations for the final evaluation, while `random_state` makes this particular split repeatable.

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,       # fraction reserved for the final, untouched test set
    stratify=y,           # preserve the target-class proportion in both subsets
    random_state=RANDOM_STATE,  # reproduce this exact split
)

print(f"Training: {X_train.shape}; test: {X_test.shape}")
print(f"Survival rate in training: {y_train.mean():.3f}")
print(f"Survival rate in test:     {y_test.mean():.3f}")

Training: (668, 7); test: (223, 7)
Survival rate in training: 0.383
Survival rate in test:     0.386


### 2.2 `fit` and `transform` with a transformer

A `StandardScaler` learns the mean and standard deviation of each numeric column from training data. Then, `transform` uses **those same learned values** for training and test data. This prevents the test set from influencing feature preparation.

In [4]:
numeric_demo = ["age", "fare"]
scaler = StandardScaler()

print("Before<<<<<")
display(pd.DataFrame(X_test[:5], columns=numeric_demo))

# fit learns the training-set mean and standard deviation for each selected feature.
scaler.fit(X_train[numeric_demo])
X_train_scaled = scaler.transform(X_train[numeric_demo])
# transform reuses those learned values; it must not fit again on the test set.
X_test_scaled = scaler.transform(X_test[numeric_demo])

print("Mean of scaled training columns:", X_train_scaled.mean(axis=0).round(2))
print("Mean of scaled test columns:    ", X_test_scaled.mean(axis=0).round(2))

print("After<<<<<")
pd.DataFrame(X_test_scaled[:5], columns=numeric_demo)

Before<<<<<


,age,fare
157,30.0,8.0500
501,21.0,7.7500
352,15.0,7.2292
82,28.0,7.7875
683,14.0,46.9000


Mean of scaled training columns: [0. 0.]
Mean of scaled test columns:     [-0.06  0.09]
After<<<<<


,age,fare
0,0.034457,-0.487691
1,-0.659352,-0.494018
2,-1.121892,-0.505000
3,-0.119723,-0.493227
4,-1.198982,0.331543


The training mean is close to zero by construction. The test mean does not need to be zero: it was transformed with training parameters, just as future passengers would be. `fit_transform(X_train)` is only a shortcut for `fit(X_train)` followed by `transform(X_train)`.

### 2.3 Encoding categories

scikit-learn models receive a numeric matrix. `OneHotEncoder` turns every category into an indicator column. `handle_unknown="ignore"` allows the pipeline to transform an unseen port or category without failing. `sparse_output=False` requests a dense array here so that its values are easy to inspect.

In [5]:
categorical_demo = ["sex", "embarked"]

display(pd.DataFrame(X_train[categorical_demo][:5]))

encoder = OneHotEncoder(
    handle_unknown="ignore",  # unseen categories become all-zero indicators instead of raising an error
    sparse_output=False,       # use a dense array for this small, inspectable example
)

X_train_encoded = encoder.fit_transform(X_train[categorical_demo])
X_test_encoded = encoder.transform(X_test[categorical_demo])

print("Learned categories:", encoder.categories_)
print("Generated columns:", encoder.get_feature_names_out(categorical_demo).tolist())
display(pd.DataFrame(X_train_encoded[:5]))

,sex,embarked
486,female,S
238,male,S
722,male,S
184,female,S
56,female,S


Learned categories: [array(['female', 'male'], dtype=object), array(['C', 'Q', 'S'], dtype=object)]
Generated columns: ['sex_female', 'sex_male', 'embarked_C', 'embarked_Q', 'embarked_S']


,0,1,2,3,4
0,1.0,0.0,0.0,0.0,1.0
1,0.0,1.0,0.0,0.0,1.0
2,0.0,1.0,0.0,0.0,1.0
3,1.0,0.0,0.0,0.0,1.0
4,1.0,0.0,0.0,0.0,1.0


## 3. `ColumnTransformer`: preprocessing by column type

In practice, calling several transformers manually is inconvenient. `ColumnTransformer` applies one sequence to each column group and joins the results. Here, numeric variables are standardized and categorical variables are one-hot encoded.

Although trees do not require scaling, using the same preprocessing in this activity keeps the comparison controlled. In a more mature experiment, preprocessing can also be a hypothesis to compare.

In [6]:
# Numeric columns use a scale-sensitive transformation; categorical columns receive indicator variables.
numeric_features = ["age", "fare", "sibsp", "parch"]
categorical_features = ["sex", "pclass", "embarked"]

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", StandardScaler(), numeric_features),  # center and scale numeric features
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),  # encode categories safely
    ],
    verbose_feature_names_out=False,  # keep output feature names compact and readable
)

X_train_ready = preprocessor.fit_transform(X_train)
X_test_ready = preprocessor.transform(X_test)

print("Original shape:   ", X_train.shape)
print("Transformed shape:", X_train_ready.shape)
print("Generated features:", preprocessor.get_feature_names_out().tolist())
display(pd.DataFrame(X_train_ready)[:5])

Original shape:    (668, 7)
Transformed shape: (668, 12)
Generated features: ['age', 'fare', 'sibsp', 'parch', 'sex_female', 'sex_male', 'pclass_1', 'pclass_2', 'pclass_3', 'embarked_C', 'embarked_Q', 'embarked_S']


,0,1,2,3,4,5,6,7,8,9,10,11
0,0.419907,1.240397,0.536677,-0.451483,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
1,-0.813532,-0.436028,-0.464720,-0.451483,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0
2,0.342817,-0.383310,-0.464720,-0.451483,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0
3,-1.969882,-0.192999,-0.464720,1.951631,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
4,-0.659352,-0.436028,-0.464720,-0.451483,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0


## 4. `Pipeline`: preprocessing + model as one object

A `Pipeline` connects steps in order. When calling `pipeline.fit(X_train, y_train)`, it fits the `preprocessor`, transforms training data, and then fits the classifier. When calling `pipeline.predict(X_test)`, it repeats only the learned transformations and returns predictions.

This reduces leakage risk and ensures that training, validation, and production use the same path. The complete pipeline can later be saved as one artifact.

In [7]:
logistic_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    # max_iter caps optimization iterations; random_state fixes any randomized internals.
    ("model", LogisticRegression(max_iter=1_000, random_state=RANDOM_STATE)),
])

logistic_pipeline.fit(X_train, y_train)
y_pred = logistic_pipeline.predict(X_test)
y_prob = logistic_pipeline.predict_proba(X_test)[:, 1]

print(f"Accuracy: {accuracy_score(y_test, y_pred):.3f}")
print(f"ROC-AUC:  {roc_auc_score(y_test, y_prob):.3f}")
pd.DataFrame({
    "observed": y_test.iloc[:10].to_numpy(),
    "predicted_class": y_pred[:10],
    "survival_probability": y_prob[:10].round(3),
}, index=X_test.index[:10])

Accuracy: 0.776
ROC-AUC:  0.842


,observed,predicted_class,survival_probability
157,0,0,0.085
501,0,1,0.763
352,0,0,0.152
82,1,1,0.710
683,0,0,0.049
84,1,1,0.866
779,1,1,0.921
515,0,0,0.301
728,0,0,0.224
734,0,0,0.278


`predict_proba` returns one column per class; `[:, 1]` selects the probability of survival. `predict` maps probability to a class using the estimator's default rule (usually a 0.5 threshold). A prediction is not, by itself, a business decision: the threshold can change when error costs differ.

## 5. Comparing algorithm families with cross-validation

We will compare three modeling hypotheses on the **training set**:

- logistic regression: linear family;
- k-nearest neighbors (k-NN): distance/instance-based family;
- random forest: tree-ensemble family.

`cross_validate` receives the complete pipeline. In each fold, preprocessing is fitted only on that fold's training portion; therefore, cross-validation does not reuse statistics from validation examples.

In [8]:
models = {
    # max_iter allows sufficient iterations for the logistic optimizer to converge.
    "Logistic regression (linear)": LogisticRegression(max_iter=1_000, random_state=RANDOM_STATE),
    # n_neighbors is the number of nearby training samples that vote for each prediction.
    "k-NN (distance)": KNeighborsClassifier(n_neighbors=15),
    "Random forest (trees)": RandomForestClassifier(
        n_estimators=300,      # number of trees averaged by the ensemble
        min_samples_leaf=3,   # minimum samples per leaf; limits overly specific splits
        random_state=RANDOM_STATE,
        n_jobs=-1,            # use all available CPU cores during fitting
    ),
}

# Five stratified folds preserve class proportions; shuffling with a seed makes folds repeatable.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
rows = []
fitted_pipelines = {}

for name, model in models.items():
    candidate = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model),
    ])
    scores = cross_validate(
        candidate, X_train, y_train, cv=cv,
        scoring={"accuracy": "accuracy", "roc_auc": "roc_auc"},
        n_jobs=-1,               # evaluate folds in parallel when resources allow
        return_train_score=False, # keep only validation metrics for this comparison
    )
    rows.append({
        "model": name,
        "cv_accuracy_mean": scores["test_accuracy"].mean(),
        "cv_accuracy_std": scores["test_accuracy"].std(),
        "cv_roc_auc_mean": scores["test_roc_auc"].mean(),
        "mean_fit_time_s": scores["fit_time"].mean(),
    })
    fitted_pipelines[name] = candidate.fit(X_train, y_train)

# Sort by validation ROC-AUC: the selection metric for this introductory experiment.
results = pd.DataFrame(rows).sort_values("cv_roc_auc_mean", ascending=False)
# DataFrame.round needs no optional HTML-styling dependency such as jinja2.
display(results.round({
    "cv_accuracy_mean": 3,
    "cv_accuracy_std": 3,
    "cv_roc_auc_mean": 3,
    "mean_fit_time_s": 3,
}))

,model,cv_accuracy_mean,cv_accuracy_std,cv_roc_auc_mean,mean_fit_time_s
2,Random forest (trees),0.799,0.013,0.868,0.880
1,k-NN (distance),0.819,0.020,0.859,0.026
0,Logistic regression (linear),0.802,0.020,0.848,0.037


The table is a first local form of experiment tracking: it documents family, metrics, variation across folds, and time. W&B will automate and enrich this record, but it does not replace sound experimental design.

### 5.1 One-time evaluation of the selected candidate

We select by mean cross-validation ROC-AUC on the training set and only now inspect the test set. In real projects, selection should also consider error costs, latency, explainability, and stability — not just one metric.

In [9]:
# The top row is the candidate selected using only cross-validation on the training data.
selected_name = results.iloc[0]["model"]
selected_pipeline = fitted_pipelines[selected_name]

test_pred = selected_pipeline.predict(X_test)
test_prob = selected_pipeline.predict_proba(X_test)[:, 1]

print(f"Candidate selected by CV: {selected_name}")
print(f"Test accuracy: {accuracy_score(y_test, test_pred):.3f}")
print(f"Test ROC-AUC:  {roc_auc_score(y_test, test_prob):.3f}")
print("\nClassification report (default threshold):")
print(classification_report(y_test, test_pred, target_names=["did not survive", "survived"]))

Candidate selected by CV: Random forest (trees)
Test accuracy: 0.794
Test ROC-AUC:  0.844

Classification report (default threshold):
                 precision    recall  f1-score   support

did not survive       0.79      0.90      0.84       137
       survived       0.79      0.63      0.70        86

       accuracy                           0.79       223
      macro avg       0.79      0.76      0.77       223
   weighted avg       0.79      0.79      0.79       223



## 6. Inspecting a trained pipeline

Because a pipeline is an estimator, its fitted steps are accessible through `named_steps`. Learned attributes usually end in `_`, such as `coef_` for logistic-regression coefficients or `feature_importances_` for the forest.

In [10]:
print(selected_pipeline)
print("\nNamed steps:", list(selected_pipeline.named_steps))

fitted_preprocessor = selected_pipeline.named_steps["preprocessor"]
print("Features that reach the model:")
print(fitted_preprocessor.get_feature_names_out().tolist())

if selected_name.startswith("Logistic"):
    coefficients = pd.Series(
        selected_pipeline.named_steps["model"].coef_[0],
        index=fitted_preprocessor.get_feature_names_out(),
    ).sort_values()
    display(coefficients.to_frame("coefficient"))
elif selected_name.startswith("Random"):
    importances = pd.Series(
        selected_pipeline.named_steps["model"].feature_importances_,
        index=fitted_preprocessor.get_feature_names_out(),
    ).sort_values(ascending=False)
    display(importances.to_frame("importance"))
else:
    print("k-NN stores training examples; it does not provide global coefficients or feature importances.")

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('numeric', StandardScaler(),
                                                  ['age', 'fare', 'sibsp',
                                                   'parch']),
                                                 ('categorical',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['sex', 'pclass',
                                                   'embarked'])],
                                   verbose_feature_names_out=False)),
                ('model',
                 RandomForestClassifier(min_samples_leaf=3, n_estimators=300,
                                        n_jobs=-1, random_state=42))])

Named steps: ['preprocessor', 'model']
Features that reach the model:
['age', 'fare', 'sibsp', 'parch', 'sex_female', 'sex_male', 'pclass_1', 'pclass_2', 'pclass_3', 'embarked_C', 'embarked_Q', 'embarked_S']


,importance
sex_male,0.210215
sex_female,0.204166
fare,0.191100
age,0.155267
pclass_3,0.071459
pclass_1,0.046010
sibsp,0.035741
parch,0.032878
pclass_2,0.017706
embarked_S,0.015293


## 7. Persisting a complete random-forest pipeline

A deployable artifact must contain both preprocessing and the trained predictor. Saving only the `RandomForestClassifier` would omit the one-hot encoding and scaling required to turn raw passenger data into model features.

The standard-library `pickle` module serializes the fitted pipeline below. Only load pickle files from trusted sources: deserializing an untrusted pickle can execute arbitrary code.

In [11]:
# This candidate was fitted on all of X_train after the cross-validation comparison.
random_forest_pipeline = fitted_pipelines["Random forest (trees)"]
artifact_path = Path("Class18_titanic_random_forest_pipeline.pkl")

# wb writes the complete fitted Pipeline: preprocessor state + random-forest trees.
with artifact_path.open("wb") as artifact_file:
    pickle.dump(random_forest_pipeline, artifact_file)

print(f"Saved complete random-forest pipeline to: {artifact_path.resolve()}")
print(f"Artifact size: {artifact_path.stat().st_size / 1_024:.1f} KiB")

Saved complete random-forest pipeline to: /home/barbon/Python/440MI/notebooks/Class18_titanic_random_forest_pipeline.pkl
Artifact size: 2888.5 KiB


### 7.1 Reloading and using the artifact

The reloaded object exposes the same `predict` and `predict_proba` interface. The assertion checks that serialization did not change its predictions for the held-out test data.

In [12]:
with artifact_path.open("rb") as artifact_file:
    loaded_pipeline = pickle.load(artifact_file)

predictions_before_save = random_forest_pipeline.predict(X_test)
predictions_after_load = loaded_pipeline.predict(X_test)
np.testing.assert_array_equal(predictions_before_save, predictions_after_load)

print("Reloaded pipeline produces identical test predictions.")
display(pd.DataFrame({
    "survival_probability": loaded_pipeline.predict_proba(X_test.iloc[:5])[:, 1].round(3),
    "predicted_class": predictions_after_load[:5],
}, index=X_test.index[:5]))

Reloaded pipeline produces identical test predictions.


,survival_probability,predicted_class
157,0.205,0
501,0.823,1
352,0.423,0
82,0.803,1
683,0.066,0


## 7. Next steps before W&B

Before automating experiment tracking, record at least:

- dataset version/source and the train/test split;
- pipeline columns, transformations, and hyperparameters;
- selection metric, results per fold, and execution cost;
- final test evaluation, performed only once for the selected candidate.

In W&B, each piece of information can become a run configuration, metric, or artifact. The object that should be preserved is the **complete pipeline**, not only the classifier: inference then receives the same preprocessing as training.

### Exercises

1. Change `n_neighbors` from 15 to 5 and 31. Do the metric and running time change?
2. Remove `fare` from `numeric_features`. What happens to cross-validation? This is a small ablation.
3. Change the selection metric to `accuracy`. Does the selected family change? Why can ROC-AUC and accuracy tell different stories?
4. In a problem where missing a survivor was much more costly than a false alert, which metric and threshold would you investigate?